In [ ]:
from pathlib import Path
import csv
import hashlib
import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone

PROTOCOL_ID = "TRKH_PRETRAINED_CLASSF_B0_20260731"
RUN_TAG = "kaggle_b0_v2"
AUTO_RESUME = False
CONFIRM_FULL = False  # Chi doi True sau khi doc metrics probe.
RUN_PROBE = not AUTO_RESUME
PROBE_MIN_MACRO_F1 = 0.55
PROBE_MIN_CLASS1_F1 = 0.30
DATA_YAML_OVERRIDE = ""  # De trong neu chi co mot data.yaml tuong thich.
DATA_ROOT_OVERRIDE = ""  # De trong neu train/val nam canh data.yaml.
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
EXPECTED_CLASS_NAMES = (
    "Xoai_Song_Chua_KhoDap",
    "Xoai_Song_ChuaNhe_CoNguyCo",
    "Xoai_Chin_NgotThanh_DeDap",
    "Xoai_ChinGia_NgotGat_KhongVanChuyen",
    "Xoai_Hu_KhongAnDuoc",
)
EXPECTED_SOURCE_COMMIT = "73c96f3d8f42e80c62ab2c4e3c0691ff81f45b77"
EXPECTED_SOURCE_TREE_SHA256 = "d7de5ed0dcd5d9c4eb6e0eeaaef2939505749a6831c7b0885cde172ec822003a"
DINO_SHA256 = "2a1ec16ae28ffa07bc0ead0241ee7df9fc26451fe6f9f839b7b3afa0a906b040"
DINO_BYTES = 86362376
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print(PROTOCOL_ID, "auto_resume=", AUTO_RESUME, "run_probe=", RUN_PROBE)


In [ ]:
# Cai dung version; khi tat Internet, dinh kem wheelhouse trong /kaggle/input.
from importlib.metadata import PackageNotFoundError, version

def ensure_locked_packages(requirements):
    pending = []
    for distribution, (module, expected) in requirements.items():
        try:
            observed = version(distribution)
        except PackageNotFoundError:
            observed = None
        if observed != expected or importlib.util.find_spec(module) is None:
            pending.append(f"{distribution}=={expected}")
    if pending:
        wheel_dirs = sorted({str(path.parent) for path in INPUT_ROOT.rglob("*.whl")})
        installed = False
        if wheel_dirs:
            command = [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-index"]
            for wheel_dir in wheel_dirs:
                command += ["--find-links", wheel_dir]
            installed = subprocess.run(command + pending, check=False).returncode == 0
        if not installed:
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *pending], check=True)
    for distribution, (_, expected) in requirements.items():
        if version(distribution) != expected:
            raise RuntimeError(f"Version lock failed: {distribution}={version(distribution)} expected={expected}")

ensure_locked_packages({
    "timm": ("timm", "1.0.27"),
    "safetensors": ("safetensors", "0.7.0"),
    "PyYAML": ("yaml", "6.0.3"),
    "Pillow": ("PIL", "11.3.0"),
    "matplotlib": ("matplotlib", "3.9.4"),
    "pytest": ("pytest", "8.4.2"),
})
print("Core dependencies ready")


In [ ]:
import yaml

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

repo_roots = sorted({path.parents[2] for path in INPUT_ROOT.rglob("trkh/recipes/pretrained_classf_b0.py")})
if len(repo_roots) != 1:
    raise RuntimeError(f"Can dung 1 source TRKH_pretrained duy nhat, tim thay: {repo_roots}")
REPO_ROOT = repo_roots[0]

def ordered_names(document):
    names = document.get("names") if isinstance(document, dict) else None
    if isinstance(names, list):
        return [str(name) for name in names]
    if isinstance(names, dict):
        try:
            keys = sorted(int(key) for key in names)
            if keys != list(range(len(keys))):
                return []
            return [str(names.get(key, names.get(str(key)))) for key in keys]
        except (TypeError, ValueError):
            return []
    return []

def resolve_override(value):
    path = Path(str(value)).expanduser()
    return path if path.is_absolute() else INPUT_ROOT / path

def data_document_compatible(document):
    if not isinstance(document, dict):
        return False
    try:
        num_classes = int(document.get("nc", 5))
    except (TypeError, ValueError):
        return False
    return bool(
        ordered_names(document) == list(EXPECTED_CLASS_NAMES)
        and document.get("train")
        and document.get("val")
        and num_classes == 5
    )

if DATA_YAML_OVERRIDE:
    data_candidates = [resolve_override(DATA_YAML_OVERRIDE)]
else:
    data_candidates = []
    for path in INPUT_ROOT.rglob("data.yaml"):
        if REPO_ROOT in path.parents:
            continue
        try:
            candidate_document = yaml.safe_load(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if data_document_compatible(candidate_document):
            data_candidates.append(path)
if len(data_candidates) != 1 or not data_candidates[0].is_file():
    raise RuntimeError(
        "Can dung dung 1 data.yaml classification-folder 5 lop tuong thich; "
        f"tim thay: {data_candidates}. Neu co nhieu file, dat DATA_YAML_OVERRIDE."
    )
DATA_YAML = data_candidates[0].resolve()
DATA_DOCUMENT = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
if not data_document_compatible(DATA_DOCUMENT):
    raise RuntimeError("DATA_YAML_OVERRIDE khong dung schema 5 lop/class order B0")

def split_path_for_root(root, value):
    raw = Path(str(value))
    candidates = [raw] if raw.is_absolute() else [root / raw]
    tail = re.split(r"[\\/]", str(value).rstrip("/\\"))[-1]
    candidates.append(root / tail)
    return next((path.resolve() for path in candidates if path.is_dir()), None)

if DATA_ROOT_OVERRIDE:
    root_candidates = [resolve_override(DATA_ROOT_OVERRIDE)]
else:
    configured_root = str(DATA_DOCUMENT.get("path", "")).strip()
    root_candidates = [DATA_YAML.parent]
    if configured_root:
        raw_root = Path(configured_root)
        root_candidates.append(raw_root if raw_root.is_absolute() else DATA_YAML.parent / raw_root)
        root_tail = re.split(r"[\\/]", configured_root.rstrip("/\\"))[-1]
        root_candidates.extend([DATA_YAML.parent / root_tail, DATA_YAML.parent.parent / root_tail])
valid_roots = []
for root in root_candidates:
    root = root.resolve()
    if split_path_for_root(root, DATA_DOCUMENT["train"]) and split_path_for_root(root, DATA_DOCUMENT["val"]):
        if root not in valid_roots:
            valid_roots.append(root)
if not valid_roots:
    raise RuntimeError("Khong tim thay thu muc train/val. Hay dat DATA_ROOT_OVERRIDE.")
DATA_ROOT = valid_roots[0]

weight_candidates = []
for path in INPUT_ROOT.rglob("*.safetensors"):
    if path.is_file() and path.stat().st_size == DINO_BYTES and sha256(path) == DINO_SHA256:
        weight_candidates.append(path)
if len(weight_candidates) != 1:
    raise RuntimeError(f"Can dung 1 DINOv3 weight dung hash/kich thuoc, tim thay: {weight_candidates}")
DINO_WEIGHT = weight_candidates[0]

sys.path.insert(0, str(REPO_ROOT))
os.environ["PYTHONPATH"] = str(REPO_ROOT)
os.environ["MPLBACKEND"] = "Agg"
os.chdir(REPO_ROOT)
if re.fullmatch(r"[0-9a-f]{40}", EXPECTED_SOURCE_COMMIT) is None:
    raise RuntimeError("Notebook release chua duoc khoa source commit")
if re.fullmatch(r"[0-9a-f]{64}", EXPECTED_SOURCE_TREE_SHA256) is None:
    raise RuntimeError("Notebook release chua duoc khoa source-tree digest")
source_digest = hashlib.sha256()
source_files = sorted((REPO_ROOT / "trkh").rglob("*.py")) + sorted((REPO_ROOT / "configs").rglob("*.yaml"))
for path in source_files:
    source_digest.update(path.relative_to(REPO_ROOT).as_posix().encode("utf-8") + b"\0")
    source_digest.update(path.read_bytes().replace(b"\r\n", b"\n"))
observed_source_tree_sha256 = source_digest.hexdigest()
if observed_source_tree_sha256 != EXPECTED_SOURCE_TREE_SHA256:
    raise RuntimeError(f"Sai source snapshot: {observed_source_tree_sha256} != {EXPECTED_SOURCE_TREE_SHA256}")
source_commit_file = REPO_ROOT / "SOURCE_COMMIT.txt"
if not source_commit_file.is_file():
    raise RuntimeError("Source archive phai co SOURCE_COMMIT.txt tai repo root")
SOURCE_COMMIT = source_commit_file.read_text(encoding="utf-8").strip().lower()
if SOURCE_COMMIT != EXPECTED_SOURCE_COMMIT:
    raise RuntimeError(f"Sai source commit: {SOURCE_COMMIT} != {EXPECTED_SOURCE_COMMIT}")
SOURCE_TREE_SHA256 = EXPECTED_SOURCE_TREE_SHA256
print("repo =", REPO_ROOT)
print("source_commit =", SOURCE_COMMIT, "source_tree_sha256 =", SOURCE_TREE_SHA256)
print("data =", DATA_YAML)
print("dino =", DINO_WEIGHT)


In [ ]:
from PIL import Image
from trkh.core.config import load_data_spec
from trkh.data.dataset import ClassificationFolderDataset
from trkh.recipes.pretrained_classf_b0 import (
    DINO_SHA256 as RECIPE_DINO_SHA256,
    EXPECTED_CLASS_NAMES as RECIPE_CLASS_NAMES,
    PROTOCOL_ID as RECIPE_PROTOCOL_ID,
)

if (
    tuple(EXPECTED_CLASS_NAMES) != tuple(RECIPE_CLASS_NAMES)
    or PROTOCOL_ID != RECIPE_PROTOCOL_ID
    or DINO_SHA256 != RECIPE_DINO_SHA256
):
    raise RuntimeError("Notebook drifted from the locked local B0 source recipe")
TRAIN_DIR = split_path_for_root(DATA_ROOT, DATA_DOCUMENT["train"])
VAL_DIR = split_path_for_root(DATA_ROOT, DATA_DOCUMENT["val"])
if TRAIN_DIR is None or VAL_DIR is None:
    raise RuntimeError("Khong resolve duoc train/val cua dataset upload")
document = dict(DATA_DOCUMENT)
document["path"] = str(DATA_ROOT)
document["train"] = str(TRAIN_DIR)
document["val"] = str(VAL_DIR)
document.pop("test", None)
DEV_YAML = WORK_ROOT / "class_f_dev_test_locked.yaml"
DEV_YAML.write_text(yaml.safe_dump(document, sort_keys=False, allow_unicode=True), encoding="utf-8")
dev_spec = load_data_spec(DEV_YAML, class_name_mode="raw", expected_num_classes=5)
if dev_spec.data_format != "classification_folder":
    raise RuntimeError(f"Can classification_folder, nhan {dev_spec.data_format}")
if dev_spec.has_test_split:
    raise RuntimeError("Test lock failed: development YAML van co test")
if tuple(dev_spec.class_names) != tuple(EXPECTED_CLASS_NAMES):
    raise RuntimeError(f"Sai class order: {dev_spec.class_names}")

split_datasets = {
    split: ClassificationFolderDataset.from_data_spec(dev_spec, split=split)
    for split in ("train", "val")
}
split_counts = {split: dataset.class_counts(5) for split, dataset in split_datasets.items()}
if any(count <= 0 for counts in split_counts.values() for count in counts):
    raise RuntimeError(f"Moi lop phai co mau trong train va val: {split_counts}")

def fingerprint_split(split, dataset):
    digest = hashlib.sha256()
    content_hashes = set()
    for sample in sorted(dataset.samples, key=lambda item: str(item.image_path).lower()):
        path = sample.image_path.resolve()
        relative = path.relative_to(dataset.root_dir.resolve()).as_posix()
        file_sha = sha256(path)
        with Image.open(path) as image:
            image.verify()
        digest.update(f"{split}\0{int(sample.label)}\0{relative}\0{file_sha}\n".encode("utf-8"))
        content_hashes.add(file_sha)
    return digest.hexdigest(), content_hashes

split_sha256 = {}
split_content_hashes = {}
for split, dataset in split_datasets.items():
    split_sha256[split], split_content_hashes[split] = fingerprint_split(split, dataset)
cross_split_duplicates = split_content_hashes["train"] & split_content_hashes["val"]
if cross_split_duplicates:
    raise RuntimeError(f"Phat hien {len(cross_split_duplicates)} anh trung byte giua train/val")
development_tree_digest = hashlib.sha256()
for split in ("train", "val"):
    development_tree_digest.update(f"{split}\0{split_sha256[split]}\n".encode("utf-8"))
DEVELOPMENT_IMAGE_TREE_SHA256 = development_tree_digest.hexdigest()
dataset_contract = {
    "schema_version": 1,
    "contract": "TRKH_KAGGLE_CLASSF_COMPATIBLE_5CLASS_V1",
    "status": "passed",
    "uploaded_data_yaml": str(DATA_YAML),
    "uploaded_data_yaml_sha256": sha256(DATA_YAML),
    "development_data_yaml": str(DEV_YAML),
    "development_data_yaml_sha256": sha256(DEV_YAML),
    "root": str(DATA_ROOT),
    "class_names": list(dev_spec.class_names),
    "split_class_counts": split_counts,
    "split_totals": {split: len(dataset) for split, dataset in split_datasets.items()},
    "split_sha256": split_sha256,
    "development_image_tree_sha256": DEVELOPMENT_IMAGE_TREE_SHA256,
    "development_image_file_count": sum(len(dataset) for dataset in split_datasets.values()),
    "cross_split_exact_duplicate_count": 0,
    "all_train_val_images_decodable": True,
    "test_key_observed_in_uploaded_yaml": "test" in DATA_DOCUMENT,
    "test_split_content_opened_or_hashed": False,
    "test_model_inference_performed": False,
    "test_metrics_read": False,
}
development_contract = dataset_contract
DATASET_CONTRACT_PATH = WORK_ROOT / "kaggle_development_dataset_contract.json"
DATASET_CONTRACT_PATH.write_text(json.dumps(dataset_contract, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Dataset-compatible contract passed; test is absent from development commands:", DEV_YAML)
print(json.dumps(dataset_contract, ensure_ascii=False, indent=2))


In [ ]:
import torch
from trkh.recipes.pretrained_classf_b0 import (
    _best_history_row,
    _preflight_model,
    build_train_args,
    run_name,
    validate_auto_resume_checkpoint,
)

if not torch.cuda.is_available():
    raise RuntimeError("Hay bat GPU Accelerator trong Kaggle Settings")
properties = torch.cuda.get_device_properties(0)
vram_gib = properties.total_memory / (1024 ** 3)
if vram_gib >= 14.5:
    MICRO_BATCH = 24
elif vram_gib >= 9.5:
    MICRO_BATCH = 16
else:
    MICRO_BATCH = 12
GRAD_ACCUM = {24: 2, 16: 3, 12: 4, 8: 6}[MICRO_BATCH]
AMP_DTYPE = "bf16" if properties.major >= 8 else "fp16"
EVAL_BATCH = 64 if vram_gib >= 14.5 else 32
os.environ["TRKH_AMP_DTYPE"] = AMP_DTYPE
os.environ.setdefault("OMP_NUM_THREADS", "4")

PROBE_TAG = f"{RUN_TAG}_probe"
FULL_TAG = f"{RUN_TAG}_full"

def stage_train_args(stage, run_tag, auto_resume=False):
    return build_train_args(
        data_yaml=DEV_YAML,
        dino_checkpoint=DINO_WEIGHT,
        output_dir=RUNS_ROOT,
        stage=stage,
        run_tag=run_tag,
        batch_size=MICRO_BATCH,
        num_workers=4,
        eval_num_workers=2,
        seed=42,
        auto_resume=auto_resume,
        source_commit=SOURCE_COMMIT,
        source_tree_sha256=SOURCE_TREE_SHA256,
        dataset_image_tree_sha256=DEVELOPMENT_IMAGE_TREE_SHA256,
    )

def training_command(stage, run_tag, auto_resume=False):
    return [sys.executable, "-m", "trkh.training.train", *stage_train_args(stage, run_tag, auto_resume)]

print(properties.name, f"{vram_gib:.1f} GiB", AMP_DTYPE, f"batch={MICRO_BATCH}x{GRAD_ACCUM}")
print("probe:", shlex.join(training_command("probe", PROBE_TAG)))
print("full:", shlex.join(training_command("full", FULL_TAG, AUTO_RESUME)))


In [ ]:
def run_checked(arguments, required=True):
    command = [str(item) for item in arguments]
    print("\n$", shlex.join(command), flush=True)
    result = subprocess.run(command, cwd=REPO_ROOT, env=os.environ.copy(), check=False)
    if required and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command)
    return result.returncode

focused_tests = [
    REPO_ROOT / "tests" / "test_pretrained_semantic_branch.py",
    REPO_ROOT / "tests" / "test_timm_classifier_model.py",
    REPO_ROOT / "tests" / "test_canonical_classf_defaults.py",
    REPO_ROOT / "tests" / "test_deploy_classification_folder.py",
    REPO_ROOT / "tests" / "test_pretrained_classf_recipe.py",
    REPO_ROOT / "tests" / "test_resume_weight_and_distillation_source.py",
    REPO_ROOT / "tests" / "test_attention_viz_headless.py",
    REPO_ROOT / "tests" / "test_mobile_onnx_quantization.py",
]
if "FOCUSED_TEST_RESULT" not in globals():
    test_returncode = run_checked([sys.executable, "-m", "pytest", "-q", *focused_tests])
    FOCUSED_TEST_RESULT = {"returncode": test_returncode, "paths": [str(path) for path in focused_tests]}

def run_stage(stage, run_tag, auto_resume=False):
    train_args = stage_train_args(stage, run_tag, auto_resume)
    run_dir = RUNS_ROOT / run_name(stage, run_tag)
    preflight_dir = RUNS_ROOT / f"preflight_classf_b0_{run_tag}"
    preflight_dir.mkdir(parents=True, exist_ok=True)
    preflight_path = preflight_dir / f"{stage}_manifest.json"
    if run_dir.exists() and any(run_dir.iterdir()) and not auto_resume:
        raise RuntimeError(f"Run da ton tai: {run_dir}; doi RUN_TAG hoac dung AUTO_RESUME cho full")
    resume_contract = None
    if auto_resume:
        last_checkpoint = run_dir / "checkpoints" / "last.pt"
        if not last_checkpoint.is_file():
            raise FileNotFoundError(f"AUTO_RESUME requires {last_checkpoint}")
        resume_contract = validate_auto_resume_checkpoint(
            last_checkpoint,
            training_data_yaml=DEV_YAML,
            expected_train_args=train_args,
        )
    model_preflight = _preflight_model(train_args)
    preflight = {
        "schema_version": 1,
        "protocol": PROTOCOL_ID,
        "adapter": "KAGGLE_CLASSF_COMPATIBLE_DATA_V1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "test_locked": True,
        "test_split_content_opened_or_hashed": False,
        "old_dataset_checkpoint_or_teacher_used": False,
        "dataset": dataset_contract,
        "source_commit": SOURCE_COMMIT,
        "source_tree_sha256": SOURCE_TREE_SHA256,
        "dino_sha256": DINO_SHA256,
        "focused_tests": FOCUSED_TEST_RESULT,
        "model_preflight": model_preflight,
        "runtime": {
            "python": sys.version,
            "torch": torch.__version__,
            "gpu": properties.name,
            "vram_gib": vram_gib,
            "amp_dtype": AMP_DTYPE,
            "micro_batch": MICRO_BATCH,
            "grad_accum": GRAD_ACCUM,
            "effective_batch": MICRO_BATCH * GRAD_ACCUM,
        },
        "resume_contract": resume_contract,
        "train_args": train_args,
    }
    preflight_path.write_text(json.dumps(preflight, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    returncode = run_checked([sys.executable, "-m", "trkh.training.train", *train_args], required=False)
    marker = {
        "schema_version": 1,
        "protocol": PROTOCOL_ID,
        "adapter": "KAGGLE_CLASSF_COMPATIBLE_DATA_V1",
        "stage": stage,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "returncode": returncode,
        "preflight_manifest": str(preflight_path),
        "dataset_image_tree_sha256": DEVELOPMENT_IMAGE_TREE_SHA256,
        "source_tree_sha256": SOURCE_TREE_SHA256,
        "dino_sha256": DINO_SHA256,
        "test_locked": True,
        "metrics": _best_history_row(run_dir),
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    marker_path = run_dir / "b0_stage_complete.json"
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    if returncode != 0:
        raise RuntimeError(f"B0 {stage} failed: {marker_path}")
    return run_dir, preflight_path, marker_path

PROBE_DIR = RUNS_ROOT / run_name("probe", PROBE_TAG)
if RUN_PROBE:
    probe_marker_path = PROBE_DIR / "b0_stage_complete.json"
    probe_was_complete_before_this_cell = probe_marker_path.is_file()
    if not probe_was_complete_before_this_cell:
        _, _, probe_marker_path = run_stage("probe", PROBE_TAG)
    probe_marker = json.loads(probe_marker_path.read_text(encoding="utf-8"))
    probe_lineage = (
        probe_marker.get("dataset_image_tree_sha256"),
        probe_marker.get("source_tree_sha256"),
        probe_marker.get("dino_sha256"),
    )
    expected_probe_lineage = (DEVELOPMENT_IMAGE_TREE_SHA256, SOURCE_TREE_SHA256, DINO_SHA256)
    if probe_lineage != expected_probe_lineage:
        raise RuntimeError(f"Probe cu sai lineage; hay doi RUN_TAG: {probe_lineage}")
    probe_preflight_path = Path(str(probe_marker.get("preflight_manifest", "")))
    if not probe_preflight_path.is_file():
        raise RuntimeError("Probe cu thieu preflight manifest")
    probe_preflight = json.loads(probe_preflight_path.read_text(encoding="utf-8"))
    if probe_preflight.get("train_args") != stage_train_args("probe", PROBE_TAG):
        raise RuntimeError("Probe cu sai official B0 train contract; hay doi RUN_TAG")
    probe_metrics = probe_marker.get("metrics", {})
    if probe_marker.get("returncode") != 0 or float(probe_metrics.get("best_macro_f1") or 0) < PROBE_MIN_MACRO_F1:
        raise RuntimeError(f"Probe gate failed: {probe_marker}")
    if float(probe_metrics.get("best_class1_f1") or 0) < PROBE_MIN_CLASS1_F1:
        raise RuntimeError(f"Probe did not learn compatible class 1: {probe_metrics}")
    print("Probe gate passed:", probe_metrics)
    if not probe_was_complete_before_this_cell or not CONFIRM_FULL:
        raise RuntimeError("Da dung sau probe. Hay xem metrics, doi CONFIRM_FULL=True va chay lai cell nay de mo full train.")

RUN_DIR, FULL_PREFLIGHT_MANIFEST, _ = run_stage("full", FULL_TAG, AUTO_RESUME)
BEST_CHECKPOINT = RUN_DIR / "checkpoints" / "best.pt"
if not BEST_CHECKPOINT.is_file() or not FULL_PREFLIGHT_MANIFEST.is_file():
    raise FileNotFoundError(f"Missing best/preflight: {BEST_CHECKPOINT}, {FULL_PREFLIGHT_MANIFEST}")
print("Best checkpoint:", BEST_CHECKPOINT)


In [ ]:
EVAL_DIR = RUN_DIR / "eval_val_full"
run_checked([
    sys.executable, "-m", "trkh.evaluation.evaluate",
    "--checkpoint", BEST_CHECKPOINT,
    "--data", DEV_YAML,
    "--class-name-mode", "raw",
    "--expected-num-classes", "5",
    "--split", "val",
    "--batch-size", EVAL_BATCH,
    "--num-workers", "2",
    "--amp",
    "--max-batches", "0",
    "--paper-name", "TRKH-DINOv3-ClassF-B0",
    "--family", "TRKH-Pretrained",
    "--output-dir", EVAL_DIR,
])
PREDICTIONS = EVAL_DIR / "predictions_detailed.csv"
if not PREDICTIONS.is_file():
    PREDICTIONS = EVAL_DIR / "predictions.csv"
if not PREDICTIONS.is_file():
    raise FileNotFoundError("Evaluate did not write predictions CSV")
METRICS_DETAIL = EVAL_DIR / "metrics_detailed.json"
if not METRICS_DETAIL.is_file():
    METRICS_DETAIL = EVAL_DIR / "metrics.json"
print(METRICS_DETAIL.read_text(encoding="utf-8")[:4000])


In [ ]:
run_checked([
    sys.executable, "-m", "trkh.tools.audit_class_confusions",
    "--predictions", PREDICTIONS,
    "--data", DEV_YAML,
    "--focus-class-index", "1",
    "--top-k-images", "32",
    "--copy-images",
    "--output-dir", RUN_DIR / "audit_class1_confusions",
])
run_checked([
    sys.executable, "-m", "trkh.tools.audit_prediction_forensics",
    "--predictions", PREDICTIONS,
    "--pairs", "0-1,1-2,1-4,2-3",
    "--focus-class-index", "1",
    "--ece-bins", "15",
    "--image-stats-mode", "foreground",
    "--max-image-stats", "512",
    "--image-stats-workers", "2",
    "--top-k-images", "32",
    "--copy-images",
    "--output-dir", RUN_DIR / "audit_prediction_forensics",
])


In [ ]:
run_checked([
    sys.executable, "-m", "trkh.evaluation.xai_audit",
    "--checkpoint", BEST_CHECKPOINT,
    "--data", DEV_YAML,
    "--split", "val",
    "--class-name-mode", "raw",
    "--expected-num-classes", "5",
    "--output-dir", RUN_DIR / "xai_val",
    "--max-cases", "24",
    "--mistake-cases", "8",
    "--low-confidence-cases", "4",
    "--close-margin-cases", "4",
    "--per-class-cases", "1",
    "--focus-class-index", "1",
    "--focus-false-positive-cases", "4",
    "--focus-false-negative-cases", "4",
    "--batch-size", EVAL_BATCH,
    "--num-workers", "2",
    "--method", "gradcam",
    "--feature-source", "patch_embed",
    "--robustness-probes",
])
run_checked([
    sys.executable, "-m", "trkh.evaluation.robustness_eval",
    "--checkpoint", BEST_CHECKPOINT,
    "--data", DEV_YAML,
    "--class-name-mode", "raw",
    "--expected-num-classes", "5",
    "--batch-size", EVAL_BATCH,
    "--num-workers", "2",
    "--max-batches", "0",
    "--num-fail-cases", "16",
    "--output-dir", RUN_DIR / "robustness_val",
])


In [ ]:
run_checked([
    sys.executable, "-m", "trkh.tools.trace_architecture",
    "--checkpoint", BEST_CHECKPOINT,
    "--data", DEV_YAML,
    "--class-name-mode", "raw",
    "--expected-num-classes", "5",
    "--image-size", "256",
    "--device", "cuda",
    "--seed", "42",
    "--output-dir", RUN_DIR / "architecture_trace",
])

# DINOv3 B0 la teacher/control, khong phai mobile model. Day chi la technical export.
deploy_code = -1
try:
    ensure_locked_packages({"onnx": ("onnx", "1.19.1")})
except Exception as error:
    print("WARNING: bo qua ONNX export vi dependency offline:", repr(error))
else:
    deploy_code = run_checked([
        sys.executable, "-m", "trkh.inference.deploy",
        "--checkpoint", BEST_CHECKPOINT,
        "--data", DEV_YAML,
        "--class-name-mode", "raw",
        "--expected-num-classes", "5",
        "--output-dir", RUN_DIR / "deploy_teacher_fp32",
        "--batch-size", EVAL_BATCH,
        "--num-workers", "2",
        "--benchmark-batch-size", "1",
        "--benchmark-warmup", "2",
        "--benchmark-runs", "2",
        "--split", "val",
        "--skip-accuracy",
        "--skip-benchmark",
        "--skip-trt-engine",
    ], required=False)
    if deploy_code:
        print("WARNING: technical ONNX export failed; checkpoint/audits remain valid. returncode=", deploy_code)


In [ ]:
full_preflight = json.loads(FULL_PREFLIGHT_MANIFEST.read_text(encoding="utf-8"))
validation_metrics = json.loads(METRICS_DETAIL.read_text(encoding="utf-8"))
validation_support = sum(int(row["support"]) for row in validation_metrics["per_class"])
with PREDICTIONS.open("r", encoding="utf-8-sig", newline="") as handle:
    prediction_rows = sum(1 for _ in csv.DictReader(handle))
expected_validation_support = int(dataset_contract["split_totals"]["val"])
if validation_support != expected_validation_support or prediction_rows != expected_validation_support:
    raise RuntimeError(
        "Full validation incomplete: "
        f"expected={expected_validation_support}, support={validation_support}, predictions={prediction_rows}"
    )
class1_metrics = validation_metrics["per_class"][1]
audit_artifacts = {
    "validation": METRICS_DETAIL,
    "predictions": PREDICTIONS,
    "class1_confusions": RUN_DIR / "audit_class1_confusions",
    "prediction_forensics": RUN_DIR / "audit_prediction_forensics",
    "xai": RUN_DIR / "xai_val" / "xai_audit_summary.json",
    "robustness": RUN_DIR / "robustness_val" / "robustness_summary.json",
    "architecture_trace": RUN_DIR / "architecture_trace" / "trace_summary.json",
}
if deploy_code == 0:
    audit_artifacts["technical_onnx_export"] = RUN_DIR / "deploy_teacher_fp32"
audit_status = {
    name: (path.is_file() or (path.is_dir() and any(path.rglob("*"))))
    for name, path in audit_artifacts.items()
}
if not all(audit_status.values()):
    raise RuntimeError(f"Required audit missing: {audit_status}")
shutil.copy2(DATA_YAML, RUN_DIR / "uploaded_data.yaml")
shutil.copy2(DEV_YAML, RUN_DIR / DEV_YAML.name)
shutil.copy2(DATASET_CONTRACT_PATH, RUN_DIR / DATASET_CONTRACT_PATH.name)
shutil.copy2(FULL_PREFLIGHT_MANIFEST, RUN_DIR / "b0_full_preflight_manifest.json")
if (PROBE_DIR / "b0_stage_complete.json").is_file():
    shutil.copy2(PROBE_DIR / "b0_stage_complete.json", RUN_DIR / "b0_probe_stage_complete.json")
evidence_roots = list(audit_artifacts.values()) + [
    BEST_CHECKPOINT,
    RUN_DIR / "uploaded_data.yaml",
    RUN_DIR / DEV_YAML.name,
    RUN_DIR / DATASET_CONTRACT_PATH.name,
    RUN_DIR / "b0_full_preflight_manifest.json",
]
evidence_files = set()
for root in evidence_roots:
    if root.is_file():
        evidence_files.add(root)
    elif root.is_dir():
        evidence_files.update(path for path in root.rglob("*") if path.is_file())
evidence_inventory = [
    {
        "path": path.relative_to(RUN_DIR).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
    for path in sorted(evidence_files)
]
EVIDENCE_INVENTORY = RUN_DIR / "evidence_sha256_inventory.json"
EVIDENCE_INVENTORY.write_text(json.dumps(evidence_inventory, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
runtime_distributions = {}
for distribution in ("torch", "torchvision", "timm", "safetensors", "numpy", "Pillow", "PyYAML", "matplotlib", "pytest", "onnx", "onnxruntime"):
    try:
        runtime_distributions[distribution] = version(distribution)
    except PackageNotFoundError:
        runtime_distributions[distribution] = None
manifest = {
    "schema_version": 1,
    "protocol": PROTOCOL_ID,
    "adapter": "KAGGLE_CLASSF_COMPATIBLE_DATA_V1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "uploaded_data_yaml_sha256": dataset_contract["uploaded_data_yaml_sha256"],
    "development_data_yaml_sha256": dataset_contract["development_data_yaml_sha256"],
    "development_image_tree_sha256": DEVELOPMENT_IMAGE_TREE_SHA256,
    "dataset_contract_sha256": sha256(DATASET_CONTRACT_PATH),
    "local_class_f_hash_or_count_compared": False,
    "source_commit": SOURCE_COMMIT,
    "source_tree_sha256": SOURCE_TREE_SHA256,
    "dino_sha256": DINO_SHA256,
    "best_checkpoint": str(BEST_CHECKPOINT),
    "best_checkpoint_sha256": sha256(BEST_CHECKPOINT),
    "validation_only": True,
    "validation_support": validation_support,
    "validation_prediction_rows": prediction_rows,
    "validation_metrics": {
        "accuracy": validation_metrics["accuracy"],
        "macro_precision": validation_metrics["macro_precision"],
        "macro_recall": validation_metrics["macro_recall"],
        "macro_f1": validation_metrics["macro_f1"],
        "class1": class1_metrics,
    },
    "audit_status": audit_status,
    "evidence_inventory": str(EVIDENCE_INVENTORY),
    "evidence_inventory_sha256": sha256(EVIDENCE_INVENTORY),
    "evidence_inventory_entries": len(evidence_inventory),
    "test_key_observed_in_uploaded_yaml": dataset_contract["test_key_observed_in_uploaded_yaml"],
    "test_split_content_opened_or_hashed": False,
    "test_pixels_used_by_model": False,
    "test_metrics_or_model_selection_used": False,
    "test_metrics_read": False,
    "old_dataset_checkpoint_or_teacher_used": False,
    "model_recipe_parity": {
        "official_builder": "trkh.recipes.pretrained_classf_b0.build_train_args",
        "source_and_dino_locked_to_local_b0": True,
        "data_identity_is_uploaded_run_specific": True,
    },
    "development_data": development_contract,
    "full_preflight_manifest_sha256": sha256(FULL_PREFLIGHT_MANIFEST),
    "train_args": full_preflight["train_args"],
    "python": sys.version,
    "runtime_distributions": runtime_distributions,
    "gpu": properties.name,
    "vram_gib": vram_gib,
    "amp_dtype": AMP_DTYPE,
    "micro_batch": MICRO_BATCH,
    "grad_accum": GRAD_ACCUM,
    "deploy_returncode": deploy_code,
}
MANIFEST = RUN_DIR / "kaggle_artifact_manifest.json"
MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
archive_base = WORK_ROOT / f"TRKH_CLASSF_B0_{RUN_TAG}_RESULTS"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Download:", archive_path)
